# OC23 Conformational-Ensemble Evaluation (AFsample2-compatible)

This notebook evaluates predicted PDB ensembles the way **AFsample2** and **AFcluster** do:
by comparing every generated model to the **experimental open and closed reference structures**
with **TM-align** (fixed `d0 = 3.5 Å`), then reporting ensemble-level accuracy.

**It does NOT use pLDDT/pTM as accuracy.** pLDDT is recorded only as a secondary column.

### Metrics produced (all standard in the field)
- **best_tm_s1 / best_tm_s2** — highest TM-score any ensemble member reaches to each reference state.
- **success** — target counts as solved if **both** states are captured (best TM > 0.8 for both), the exact
  AFsample2 criterion. We also sweep the threshold (0.5–1.0) to reproduce their success-vs-threshold (AUC) curve.
- **fill_ratio** — AFsample2's diversity metric: fraction of the open→closed path (100 bins) populated by at least one model.
- **diversity plot** — the TM-to-state1 vs TM-to-state2 scatter, per target.

### Published OC23 baselines to compare against (fraction of 23 targets with BOTH states TM>0.8)
| Method | Success |
|---|---|
| **AFsample2** | **78.3%** |
| SPEACH_AF | 73.9% |
| MSAsubsample | 69.6% |
| AFsample | 56.5% |
| AFvanilla | 47.8% |
| AFcluster | 47.8% |

Source: Kalakoti & Wallner, *Communications Biology* 8:373 (2025), Figs. 3, 4 and Methods.


## 1. Install TM-align
We compile the official TM-align (same tool AFsample2 uses). If the compile URL ever changes,
`conda install -c bioconda tmalign` is a fallback.

In [1]:
# Compile official TM-align
!apt-get -qq install -y zstd g++ > /dev/null 2>&1
import os, subprocess
if not os.path.exists('TMalign'):
    !wget -q https://zhanggroup.org/TM-align/TMalign.cpp -O TMalign.cpp
    !g++ -O3 -ffast-math -o TMalign TMalign.cpp
    !chmod +x TMalign
# make it callable as 'TMalign'
os.environ['PATH'] = os.getcwd() + ':' + os.environ['PATH']
print(subprocess.run(['./TMalign','-h'],capture_output=True,text=True).stdout[:200] or "TMalign ready")


The system cannot find the path specified.
'wget' is not recognized as an internal or external command,
operable program or batch file.
cc1plus.exe: fatal error: TMalign.cpp: No such file or directory
compilation terminated.
'chmod' is not recognized as an internal or external command,
operable program or batch file.


FileNotFoundError: [WinError 2] The system cannot find the file specified

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the OC23 reference structures (open/closed PDBs)
Downloads the AFsample2 input dataset from Zenodo and extracts **only** the OC23 reference PDBs
and `filtered_dict.pickle` (which maps each protein to its two reference states). ~226 MB.

Run once; it caches to Drive so you never repeat it.

In [ ]:
import os, subprocess, pickle, pprint

REF_DIR = '/content/drive/MyDrive/Research/OC23_references'
os.makedirs(REF_DIR, exist_ok=True)

tar_path = os.path.join(REF_DIR, 'input_datasets.tar.zst')
if not os.path.exists(os.path.join(REF_DIR,'input_datasets','oc23','filtered_dict.pickle')):
    if not os.path.exists(tar_path):
        print("Downloading input_datasets.tar.zst from Zenodo (~226 MB)...")
        !wget -q https://zenodo.org/records/14534088/files/input_datasets.tar.zst -O "$tar_path"
    print("Extracting OC23 references...")
    subprocess.run(['tar','--use-compress-program=unzstd','-xvf',tar_path,
                    '-C',REF_DIR,
                    'input_datasets/oc23/pdbs',
                    'input_datasets/oc23/filtered_dict.pickle',
                    'input_datasets/oc23/fastas'], check=False)
OC23_DIR = os.path.join(REF_DIR,'input_datasets','oc23')
PDBS_DIR = os.path.join(OC23_DIR,'pdbs')
print("Reference PDBs available:", len(os.listdir(PDBS_DIR)) if os.path.isdir(PDBS_DIR) else 'MISSING')


### 3b. Inspect `filtered_dict.pickle` and build the protein → (state1, state2) map
The pickle structure isn't documented, so we print it first, then extract the two reference
PDB filenames for each protein. **Check the printout** and adjust `build_ref_map` if the keys differ.

In [ ]:
import pickle, os, pprint
with open(os.path.join(OC23_DIR,'filtered_dict.pickle'),'rb') as fh:
    filtered = pickle.load(fh)

print("Type:", type(filtered))
if isinstance(filtered, dict):
    keys = list(filtered.keys())
    print("Num entries:", len(keys))
    print("Example keys:", keys[:5])
    print("\nExample value for", keys[0], ":")
    pprint.pprint(filtered[keys[0]])
else:
    pprint.pprint(filtered[:3] if hasattr(filtered,'__getitem__') else filtered)


In [ ]:
# Build {protein_id: (path_state1, path_state2)} from the pickle.
# ADAPT the field names below to match the printout in the previous cell.
import glob, os

def find_pdb(name):
    """Locate a reference pdb file by (partial) name inside PDBS_DIR."""
    if name is None: return None
    name = str(name)
    cands = glob.glob(os.path.join(PDBS_DIR, f'*{name}*'))
    return cands[0] if cands else None

def build_ref_map(filtered):
    ref_map = {}
    for prot, info in filtered.items():
        s1 = s2 = None
        if isinstance(info, dict):
            # try common key names — adjust after seeing the printout
            for k in ['state1','open','pdb1','ref1','s1','apo']:
                if k in info: s1 = info[k]; break
            for k in ['state2','closed','pdb2','ref2','s2','holo']:
                if k in info: s2 = info[k]; break
        elif isinstance(info,(list,tuple)) and len(info)>=2:
            s1, s2 = info[0], info[1]
        p1, p2 = find_pdb(s1), find_pdb(s2)
        if p1 and p2:
            ref_map[str(prot)] = (p1, p2)
    return ref_map

ref_map = build_ref_map(filtered)
print(f"Built reference map for {len(ref_map)} / {len(filtered)} proteins")
for k in list(ref_map)[:5]:
    print(k, '->', [os.path.basename(x) for x in ref_map[k]])
# If this is empty or wrong, hard-code it, e.g.:
# ref_map = {'Q18A65': (f'{PDBS_DIR}/xxxx_A.pdb', f'{PDBS_DIR}/yyyy_A.pdb'), ...}


## 4. Configuration — point to  predicted ensembles

In [ ]:
# Folder that contains one subfolder per protein with the generated PDBs.
# /content/drive/MyDrive/Research/dropout_predictions/OC23/<jobname>/...
PRED_ROOT = '/content/drive/MyDrive/Research/dropout_predictions/OC23'
OUT_DIR   = '/content/drive/MyDrive/Research/evaluation_OC23'
os.makedirs(OUT_DIR, exist_ok=True)

SUCCESS_TM = 0.8   # AFsample2 primary success threshold (both states)
NCPU       = 4

# Map each prediction folder -> protein id used in ref_map.
# my prediction folders are named like 'Q18A65_ab12c'. We strip the hash to get the UniProt id.
import re
def folder_to_protein(folder_name):
    return re.split(r'[_\.]', folder_name)[0]   # 'Q18A65_ab12c' -> 'Q18A65'

pred_folders = [f for f in os.listdir(PRED_ROOT) if os.path.isdir(os.path.join(PRED_ROOT,f))]
print("Prediction folders found:", pred_folders)


## 5. TM-align wrapper (identical parsing to AFsample2's `analyse_models.py`)

In [ ]:
import subprocess, glob, numpy as np, pandas as pd
from pathlib import Path
from multiprocessing import Pool
from Bio.PDB import PDBParser

def tmalign(model, reference, tmalign_path='./TMalign'):
    """Return TM-score normalized with user-specified d0=3.5 (AFsample2 convention)."""
    try:
        r = subprocess.run([tmalign_path, model, reference, '-d', '3.5'],
                           capture_output=True, text=True, check=True)
    except subprocess.CalledProcessError as e:
        return None
    tm = None
    for line in r.stdout.splitlines():
        if 'user-specified d0' in line:
            try:
                seg = line.split('=')[1].split(' ')
                tm = float(seg[1])
            except (IndexError, ValueError):
                pass
    return tm

def mean_plddt(pdb_file):
    """Mean per-residue B-factor = mean pLDDT for AF2 PDBs."""
    try:
        st = PDBParser(QUIET=True).get_structure('s', pdb_file)
        vals = [np.mean([a.get_bfactor() for a in res]) for chain in st[0] for res in chain]
        return float(np.mean(vals))
    except Exception:
        return np.nan

def _score_pair(args):
    model, ref = args
    return tmalign(model, ref)


## 6. Score every model against both reference states

In [ ]:
def list_models(protein_folder):
    # grab all generated PDBs recursively (ColabFold names contain 'unrelaxed'/'rank')
    pdbs = glob.glob(os.path.join(protein_folder, '**', '*.pdb'), recursive=True)
    return sorted(pdbs)

per_model_rows = []
per_target_rows = []

for folder in pred_folders:
    prot = folder_to_protein(folder)
    if prot not in ref_map:
        print(f"[skip] {folder}: no reference for protein '{prot}'")
        continue
    ref1, ref2 = ref_map[prot]
    models = list_models(os.path.join(PRED_ROOT, folder))
    if not models:
        print(f"[skip] {folder}: no PDBs found"); continue
    print(f"[{prot}] scoring {len(models)} models ...")

    with Pool(NCPU) as pool:
        tm1 = pool.map(_score_pair, [(m, ref1) for m in models])
        tm2 = pool.map(_score_pair, [(m, ref2) for m in models])
    plddt = [mean_plddt(m) for m in models]

    # TM between the two experimental states (needed for fill-ratio)
    tm_ref = tmalign(ref1, ref2)

    for m, a, b, c in zip(models, tm1, tm2, plddt):
        per_model_rows.append({'protein':prot,'model':os.path.basename(m),
                               'tm_s1':a,'tm_s2':b,'plddt':c})
    dfp = pd.DataFrame({'tm_s1':tm1,'tm_s2':tm2})
    dfp = dfp.dropna()
    best1, best2 = dfp['tm_s1'].max(), dfp['tm_s2'].max()
    per_target_rows.append({'protein':prot,'n_models':len(dfp),
                            'tm_ref_open_closed':tm_ref,
                            'best_tm_s1':best1,'best_tm_s2':best2,
                            'min_best':min(best1,best2),
                            'success': bool(best1>SUCCESS_TM and best2>SUCCESS_TM)})

model_df  = pd.DataFrame(per_model_rows)
target_df = pd.DataFrame(per_target_rows)
model_df.to_csv(os.path.join(OUT_DIR,'per_model_scores.csv'), index=False)
target_df.to_csv(os.path.join(OUT_DIR,'per_target_summary.csv'), index=False)
target_df


## 7. Fill-ratio (AFsample2 diversity metric)

In [ ]:
def fill_ratio(tm_s1, tm_s2, tm_ref, nbins=100):
    """Replicates AFsample2 Methods: fraction of open->closed path bins populated,
    with parabolic weighting emphasizing the two ends."""
    import numpy as np
    tm_s1 = np.asarray(tm_s1); tm_s2 = np.asarray(tm_s2)
    # Q1 = models close to both references
    mask = (tm_s1 >= tm_ref) & (tm_s2 >= tm_ref)
    x, y = tm_s1[mask], tm_s2[mask]
    m, q = -1.0, 1.0 + tm_ref
    L = np.sqrt(2) * (1.0 - tm_ref)
    if L <= 0: return 0.0
    binw = L / nbins
    k_all = np.arange(nbins)
    w = 1 + 16*((k_all/(nbins-1)) - 0.5)**2
    populated = set()
    for xi, yi in zip(x, y):
        xp = (xi + m*yi - m*q)/(m*m+1); yp = m*xp + q
        s = np.sqrt((xp-tm_ref)**2 + (yp-1.0)**2)
        k = int(np.clip(np.floor(s/binw), 0, nbins-1))
        populated.add(k)
    return float(sum(w[k] for k in populated) / w.sum())

fr = []
for _, r in target_df.iterrows():
    sub = model_df[model_df.protein==r.protein].dropna(subset=['tm_s1','tm_s2'])
    fr.append(fill_ratio(sub.tm_s1, sub.tm_s2, r.tm_ref_open_closed))
target_df['fill_ratio'] = fr
target_df.to_csv(os.path.join(OUT_DIR,'per_target_summary.csv'), index=False)
target_df[['protein','best_tm_s1','best_tm_s2','min_best','success','fill_ratio']]


## 8. Diversity plots (TM-to-state1 vs TM-to-state2), per target

In [ ]:
import matplotlib.pyplot as plt
for prot in target_df.protein:
    sub = model_df[model_df.protein==prot].dropna(subset=['tm_s1','tm_s2'])
    r = target_df[target_df.protein==prot].iloc[0]
    plt.figure(figsize=(5,5))
    sc = plt.scatter(sub.tm_s1, sub.tm_s2, c=sub.plddt, cmap='viridis', s=18, vmin=20, vmax=100)
    plt.colorbar(sc, label='mean pLDDT')
    plt.axhline(0.8, ls='--', c='gray', lw=0.8); plt.axvline(0.8, ls='--', c='gray', lw=0.8)
    plt.xlabel('TM-score to state 1'); plt.ylabel('TM-score to state 2')
    plt.title(f'{prot}  |  best=({r.best_tm_s1:.2f},{r.best_tm_s2:.2f})  '
              f'{"SOLVED" if r.success else "not solved"}')
    plt.xlim(0,1); plt.ylim(0,1); plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f'diversity_{prot}.png'), dpi=130)
    plt.show()


## 9. Summary + comparison to published baselines
This is the number that matters for your thesis: **your success rate vs the published methods.**
Because you generate ~100 models vs their 1000, also report success **per 100 GPU-forward-passes**
to make the efficiency argument explicit.

In [ ]:
n = len(target_df)
solved = int(target_df.success.sum())
your_success = 100.0*solved/n if n else 0.0

baselines = {'AFsample2 (1000)':78.3,'SPEACH_AF (1000)':73.9,'MSAsubsample (1000)':69.6,
             'AFsample (1000)':56.5,'AFvanilla':47.8,'AFcluster (1000)':47.8}
print(f"YOUR SYSTEM:  {solved}/{n} targets solved (both states TM>0.8) = {your_success:.1f}%")
print(f"  mean best_tm_s1 = {target_df.best_tm_s1.mean():.3f}")
print(f"  mean best_tm_s2 = {target_df.best_tm_s2.mean():.3f}")
print(f"  mean fill_ratio = {target_df.fill_ratio.mean():.3f}")
print("\nPublished OC23 baselines (fraction of 23 targets, both states TM>0.8):")
for k,v in baselines.items(): print(f"  {k:22s} {v:5.1f}%")

import matplotlib.pyplot as plt
labels = ['YOURS (100)'] + list(baselines.keys())
vals   = [your_success] + list(baselines.values())
colors = ['crimson'] + ['steelblue']*len(baselines)
plt.figure(figsize=(9,4))
plt.bar(labels, vals, color=colors)
plt.ylabel('% targets both states TM>0.8'); plt.xticks(rotation=35, ha='right')
plt.title('OC23 success rate — your system vs published methods')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,'comparison_success.png'), dpi=130); plt.show()


---
### How to read your result honestly
- **If your success rate is at/above ~70%** on the targets you ran, you are competitive with SPEACH_AF/MSAsubsample and near AFsample2 — at **1/10 the sampling**. That is a positive, publishable efficiency result.
- **If it is lower**, the diversity plots tell you *why*: points stuck only in one corner = not enough perturbation (raise masking / use all 5 models); points scattered with low TM to both = over-perturbation on shallow sub-MSAs (your adaptive-masking fix).
- Either way you now have the **field-standard number**, not pLDDT — which is what a thesis committee and any reviewer will ask for.
